In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import random
import re

BASE_URL = "https://www.yelp.nl/biz/simonis-den-haag-5"

# Setup driver (FAST)
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")

    # Disable images/fonts
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.managed_default_content_settings.fonts": 2
    }
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)

    # Block network resources
    driver.execute_cdp_cmd("Network.enable", {})
    driver.execute_cdp_cmd(
        "Network.setBlockedURLs",
        {
            "urls": [
                "*.jpg", "*.jpeg", "*.png", "*.gif", "*.webp",
                "*.woff", "*.woff2", "*.ttf",
                "*.mp4", "*.webm"
            ]
        }
    )

    return driver


# Accept cookies
def accept_cookies(driver):
    try:
        wait = WebDriverWait(driver, 5)
        btn = wait.until(EC.element_to_be_clickable(
            (By.XPATH, '//button//span[contains(text(),"Accept")]')
        ))
        btn.click()
        print("Cookies accepted")
    except:
        pass


# Extract reviews
def get_reviews(driver, start):
    url = f"{BASE_URL}?start={start}"
    driver.get(url)

    wait = WebDriverWait(driver, 15)

    accept_cookies(driver)

    # Wait for review text to load
    try:
        wait.until(EC.presence_of_element_located(
            (By.XPATH, '//span[@lang]')
        ))
    except:
        print("No reviews found.")
        return []

    time.sleep(2)

    # Scroll to trigger lazy loading
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    # Get all review texts
    text_elements = driver.find_elements(By.XPATH, '//span[@lang]')

    print(f"Found {len(text_elements)} text elements")

    reviews = []

    for text_el in text_elements:
        # TEXT
        try:
            text = text_el.text
        except:
            text = None

        # RATING
        try:
            parent = text_el.find_element(
                By.XPATH,
                './ancestor::div[.//*[@role="img" and contains(@aria-label, "ster")]][1]'
            )

            rating_el = parent.find_element(
                By.XPATH,
                './/*[@role="img" and contains(@aria-label, "ster")]'
            )

            rating = rating_el.get_attribute("aria-label")

            import re
            match = re.search(r'[\d.,]+', rating)
            rating_number = float(match.group().replace(',', '.')) if match else None

        except Exception as e:
            print("Rating error:", e)
            rating_number = None

        reviews.append({
            "text": text,
            "rating": rating_number
        })

    return reviews


# Main scraper loop
def scrape_all_reviews(max_pages=20):
    driver = setup_driver()
    all_reviews = []

    try:
        for page in range(max_pages):
            start = page * 10
            print(f"\nScraping reviews starting at {start}...")

            reviews = get_reviews(driver, start)

            if not reviews:
                print("Stopping: no reviews found.")
                break

            all_reviews.extend(reviews)

            time.sleep(random.uniform(2, 4))

    finally:
        driver.quit()

    return all_reviews


# Run script
if __name__ == "__main__":
    reviews = scrape_all_reviews(max_pages=30)

    print(f"\n✅ Collected {len(reviews)} reviews\n")

    for r in reviews[:5]:
        print(r)


Scraping reviews starting at 0...
Found 10 text elements

Scraping reviews starting at 10...
Found 10 text elements

Scraping reviews starting at 20...
Found 6 text elements

Scraping reviews starting at 30...
No reviews found.
Stopping: no reviews found.

✅ Collected 26 reviews

{'text': "The portion here are huge! We got the kibbeling, lekkerbekje (i.e. Dutch verison of fish and chips), and two side order of fries. Honestly, we should have gotten one order of fries to share since both fried fish entries were already big portions. There's also a section where you can get a bunch of dipping sauces for your fried fish. From the two fried fishes, I enjoyed eating the kibbeling more than the lekkerbekje.", 'rating': 4.0}
{'text': "If you find your self in Scheveningen this is the place you have to visit to eat.\n\nPlenty of place outside and inside, and a large menu to choose from. When you are in the Netherlands you need to eat fish, they have all kinds of sorts you will not find in the

In [8]:
for r in reviews:
    print(r)

{'text': "The portion here are huge! We got the kibbeling, lekkerbekje (i.e. Dutch verison of fish and chips), and two side order of fries. Honestly, we should have gotten one order of fries to share since both fried fish entries were already big portions. There's also a section where you can get a bunch of dipping sauces for your fried fish. From the two fried fishes, I enjoyed eating the kibbeling more than the lekkerbekje.", 'rating': 4.0}
{'text': "If you find your self in Scheveningen this is the place you have to visit to eat.\n\nPlenty of place outside and inside, and a large menu to choose from. When you are in the Netherlands you need to eat fish, they have all kinds of sorts you will not find in the US, so you don't want to miss out. The portions are generous, one portion of kibbeling feeds 2 easily. I started of with the Simonis soup a small bowl with an amazingly tasty and rich seafood soup. The service is fast , and the experience is priceless. Enjoy when you are in Scheve